# Arsenal 2024/25 Data Preparation

## Purpose

This notebook builds the clean player-match dataset that all later notebooks use. It performs no analysis, only data loading, cleaning, validation, and export.

## Outputs

This notebook saves two files to `/processed`:
- `arsenal_player_match_master.csv` — one row per player-match, all seasons combined
- `arsenal_player_rest_days.csv` — one row per player-match with rest days calculated

## 1. Project Setup

In [34]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Define data paths
DATA_BASE = Path('/home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025')
ARSENAL_PATH = DATA_BASE / 'arsenal_2024_2025'
PLAYER_STATS_PATH = ARSENAL_PATH / 'player_stats'
MATCH_REPORTS_PATH = ARSENAL_PATH / 'match_reports'
PROCESSED_PATH = ARSENAL_PATH / 'processed'

# Create processed directory if it doesn't exist
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print(f"Data base path: {DATA_BASE}")
print(f"Arsenal data path: {ARSENAL_PATH}")
print(f"Player stats path: {PLAYER_STATS_PATH}")
print(f"Match reports path: {MATCH_REPORTS_PATH}")
print(f"Processed output path: {PROCESSED_PATH}")

Data base path: /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025
Arsenal data path: /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025
Player stats path: /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/player_stats
Match reports path: /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/match_reports
Processed output path: /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/processed


## 2. Load Raw Arsenal 2024/25 Data

In [35]:
# ============================================================================
# 1.3 LOAD MATCH-LEVEL DATA: LINEUPS, PLAYER STATS, GOALKEEPER STATS
# ============================================================================

player_stats_file = PLAYER_STATS_PATH / 'arsenal_2024_2025_players_all_competitions.csv'
player_stats_season = pd.read_csv(player_stats_file)

match_reports_folders = sorted([d for d in MATCH_REPORTS_PATH.iterdir() if d.is_dir()])

lineups_list = []
player_match_stats_list = []
goalkeeper_stats_list = []

print(f"Season-level player stats file: {player_stats_file.name}")
print(f"Found {len(match_reports_folders)} match report folders")

for match_folder in match_reports_folders:
    match_name = match_folder.name
    
    lineups_file = match_folder / f'{match_name}_lineups.csv'
    player_stats_match_file = match_folder / f'{match_name}_player_stats.csv'
    gk_stats_file = match_folder / f'{match_name}_goalkeeper_stats.csv'
    
    if lineups_file.exists():
        lineups_list.append(pd.read_csv(lineups_file))
    
    if player_stats_match_file.exists():
        player_match_stats_list.append(pd.read_csv(player_stats_match_file))
    
    if gk_stats_file.exists():
        goalkeeper_stats_list.append(pd.read_csv(gk_stats_file))

lineups_master = pd.concat(lineups_list, ignore_index=True) if lineups_list else pd.DataFrame()
player_match_master = pd.concat(player_match_stats_list, ignore_index=True) if player_match_stats_list else pd.DataFrame()
goalkeeper_master = pd.concat(goalkeeper_stats_list, ignore_index=True) if goalkeeper_stats_list else pd.DataFrame()

print(f"\nMatches loaded: {len(match_reports_folders)}")
print(f"Lineups shape: {lineups_master.shape}")
print(f"Player match stats shape: {player_match_master.shape}")
print(f"Goalkeeper stats shape: {goalkeeper_master.shape}")

Season-level player stats file: arsenal_2024_2025_players_all_competitions.csv
Found 58 match report folders

Matches loaded: 58
Lineups shape: (1136, 14)
Player match stats shape: (866, 33)
Goalkeeper stats shape: (58, 18)


## 3. Clean and Standardize Column Names

In [36]:
# ============================================================================
# 1.7 STANDARDIZE COLUMN NAMES AND DATE TYPES
# ============================================================================

print("=" * 100)
print("1.7 STANDARDIZE COLUMN NAMES AND DATE TYPES")
print("=" * 100)

# ----------------------------------------------------------------------------
# 1.7.1 Helper function to clean column names
# ----------------------------------------------------------------------------

def clean_column_names(df):
    """
    Standardize dataframe column names:
    - lowercase
    - strip spaces
    - replace spaces with underscores
    - replace repeated underscores
    """
    df = df.copy()
    
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("%", "pct", regex=False)
        .str.replace(r"_+", "_", regex=True)
    )
    
    return df


# ----------------------------------------------------------------------------
# 1.7.2 Apply column name cleaning
# ----------------------------------------------------------------------------

lineups_master = clean_column_names(lineups_master)
player_match_master_full = clean_column_names(player_match_master_full)
player_stats = clean_column_names(player_stats)

print("✓ Column names standardized")


# ----------------------------------------------------------------------------
# 1.7.3 Convert date columns to datetime
# ----------------------------------------------------------------------------

dataframes_to_check = {
    "lineups_master": lineups_master,
    "player_match_master_full": player_match_master_full
}

for df_name, df in dataframes_to_check.items():
    print("\n" + "-" * 100)
    print(f"Checking date column in: {df_name}")
    print("-" * 100)
    
    if "date" in df.columns:
        original_missing = df["date"].isna().sum()
        
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        
        converted_missing = df["date"].isna().sum()
        failed_conversions = converted_missing - original_missing
        
        print("✓ Date column found and converted")
        print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
        print(f"  Missing dates before conversion: {original_missing}")
        print(f"  Missing dates after conversion:  {converted_missing}")
        
        if failed_conversions > 0:
            print(f"  ⚠️ Failed date conversions: {failed_conversions}")
        else:
            print("  ✓ No failed date conversions")
    
    else:
        print(f"⚠️ No 'date' column found in {df_name}")


# ----------------------------------------------------------------------------
# 1.7.4 Confirm changes were applied
# ----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("COLUMN STANDARDIZATION SUMMARY")
print("=" * 100)

print("\nLineups columns:")
print(lineups_master.columns.tolist())

print("\nPlayer match columns:")
print(player_match_master_full.columns.tolist())

print("\nSeason player stats columns:")
print(player_stats.columns.tolist())


# ----------------------------------------------------------------------------
# 1.7.5 Basic shape check after standardization
# ----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("DATAFRAME SHAPES AFTER STANDARDIZATION")
print("=" * 100)

print(f"lineups_master shape:           {lineups_master.shape}")
print(f"player_match_master_full shape: {player_match_master_full.shape}")
print(f"player_stats shape:             {player_stats.shape}")


# ----------------------------------------------------------------------------
# 1.7.6 Check required columns for downstream analysis
# ----------------------------------------------------------------------------

required_player_match_cols = [
    "match_id",
    "date",
    "competition",
    "venue",
    "opponent",
    "team",
    "player",
    "pos",
    "min"
]

required_lineup_cols = [
    "match_id",
    "date",
    "team",
    "player"
]

print("\n" + "=" * 100)
print("REQUIRED COLUMN CHECK")
print("=" * 100)

missing_player_cols = [
    col for col in required_player_match_cols 
    if col not in player_match_master_full.columns
]

missing_lineup_cols = [
    col for col in required_lineup_cols 
    if col not in lineups_master.columns
]

if len(missing_player_cols) == 0:
    print("✓ player_match_master_full contains all required columns")
else:
    print("⚠️ player_match_master_full is missing required columns:")
    print(missing_player_cols)

if len(missing_lineup_cols) == 0:
    print("✓ lineups_master contains all required columns")
else:
    print("⚠️ lineups_master is missing required columns:")
    print(missing_lineup_cols)


# ----------------------------------------------------------------------------
# 1.7.7 Final preview
# ----------------------------------------------------------------------------

print("\n" + "=" * 100)
print("PREVIEW OF STANDARDIZED PLAYER-MATCH DATA")
print("=" * 100)

preview_cols = [
    col for col in [
        "match_id",
        "date",
        "competition",
        "venue",
        "opponent",
        "team",
        "player",
        "pos",
        "min"
    ]
    if col in player_match_master_full.columns
]

print(player_match_master_full[preview_cols].head(10).to_string(index=False))

print("\n✓ Section 1.7 completed successfully")

1.7 STANDARDIZE COLUMN NAMES AND DATE TYPES
✓ Column names standardized

----------------------------------------------------------------------------------------------------
Checking date column in: lineups_master
----------------------------------------------------------------------------------------------------
✓ Date column found and converted
  Date range: 2024-08-17 00:00:00 to 2025-05-25 00:00:00
  Missing dates before conversion: 0
  Missing dates after conversion:  0
  ✓ No failed date conversions

----------------------------------------------------------------------------------------------------
Checking date column in: player_match_master_full
----------------------------------------------------------------------------------------------------
✓ Date column found and converted
  Date range: 2024-08-17 00:00:00 to 2025-05-25 00:00:00
  Missing dates before conversion: 0
  Missing dates after conversion:  0
  ✓ No failed date conversions

COLUMN STANDARDIZATION SUMMARY

Lineups

## 4. Load Player Stats, Match Reports, Lineups, Goalkeeper Stats

In [37]:
# Display sample data from each source
print("="*80)
print("SAMPLE DATA FROM EACH SOURCE")
print("="*80)

print("\n1. SEASON-LEVEL PLAYER STATS (player_stats):")
print(f"   Shape: {player_stats.shape}")
print(f"\n   First 5 rows:")
display_cols = ['player', 'pos', 'mp', 'starts', 'min']
if all(col in player_stats.columns for col in display_cols):
    print(player_stats[display_cols].head())
else:
    print(player_stats.head())

print("\n2. LINEUPS (lineups_master):")
print(f"   Shape: {lineups_master.shape}")
print(f"   Unique matches: {lineups_master['match_id'].nunique() if 'match_id' in lineups_master.columns else 'N/A'}")
print(f"\n   First 5 rows:")
print(lineups_master.head())

print("\n3. PLAYER MATCH STATS (player_match_master_full):")
print(f"   Shape: {player_match_master_full.shape}")
print(f"   Unique matches: {player_match_master_full['match_id'].nunique() if 'match_id' in player_match_master_full.columns else 'N/A'}")
print(f"   Unique players: {player_match_master_full['player'].nunique() if 'player' in player_match_master_full.columns else 'N/A'}")
print(f"\n   First 5 rows (first 10 columns):")
print(player_match_master_full.iloc[:5, :10])

SAMPLE DATA FROM EACH SOURCE

1. SEASON-LEVEL PLAYER STATS (player_stats):
   Shape: (42, 22)

   First 5 rows:
           player    pos  mp  starts    min
0      David Raya     GK  55      55  4,980
1  William Saliba     DF  51      50  4,464
2     Declan Rice     MF  52      48  4,116
3   Thomas Partey  MF,DF  52      45  3,940
4  Jurriën Timber     DF  48      42  3,666

2. LINEUPS (lineups_master):
   Shape: (1136, 14)
   Unique matches: 55

   First 5 rows:
   match_id                                   match_report_url       date  \
0  c0e3342a  https://fbref.com/en/matches/c0e3342a/Arsenal-... 2024-08-17   
1  c0e3342a  https://fbref.com/en/matches/c0e3342a/Arsenal-... 2024-08-17   
2  c0e3342a  https://fbref.com/en/matches/c0e3342a/Arsenal-... 2024-08-17   
3  c0e3342a  https://fbref.com/en/matches/c0e3342a/Arsenal-... 2024-08-17   
4  c0e3342a  https://fbref.com/en/matches/c0e3342a/Arsenal-... 2024-08-17   

      competition        round venue opponent result  \
0  Premier Lea

## 4.5 Data Quality Validations

Before merging, validate the raw data for duplicates, coverage, and completeness.

In [38]:
# ============================================================================
# 1.8 UNIQUE PLAYER-MATCH VALIDATION
# ============================================================================

print("=" * 100)
print("UNIQUE PLAYER-MATCH VALIDATION")
print("=" * 100)

required_cols = {"match_id", "player"}

if required_cols.issubset(player_match_master_full.columns):
    duplicates = player_match_master_full.duplicated(
        subset=["match_id", "player"],
        keep=False
    )
    
    print(f"\nTotal rows: {len(player_match_master_full)}")
    print(f"Duplicate player-match rows: {duplicates.sum()}")
    
    if duplicates.sum() > 0:
        print("\nExample duplicate rows:")
        cols_to_show = [
            c for c in ["match_id", "date", "player", "team", "pos", "min"]
            if c in player_match_master_full.columns
        ]
        print(
            player_match_master_full.loc[duplicates, cols_to_show]
            .sort_values(["match_id", "player"])
            .head(20)
            .to_string(index=False)
        )
    else:
        print("✓ Each player appears once per match.")
else:
    print("⚠️ Cannot validate duplicates: match_id/player missing.")

UNIQUE PLAYER-MATCH VALIDATION

Total rows: 866
Duplicate player-match rows: 0
✓ Each player appears once per match.


In [39]:
# ============================================================================
# 1.9 MATCH COVERAGE VALIDATION
# ============================================================================

print("\n" + "=" * 100)
print("MATCH COVERAGE VALIDATION")
print("=" * 100)

n_match_folders = len(match_reports_folders)
n_matches_player_stats = player_match_master_full["match_id"].nunique()
n_matches_lineups = lineups_master["match_id"].nunique()

print(f"\nMatch report folders loaded: {n_match_folders}")
print(f"Unique matches in player stats: {n_matches_player_stats}")
print(f"Unique matches in lineups: {n_matches_lineups}")

if n_matches_player_stats == n_match_folders:
    print("✓ Player stats cover all match folders.")
else:
    print(f"⚠️ Player stats: expected {n_match_folders}, got {n_matches_player_stats}")

if n_matches_lineups == n_match_folders:
    print("✓ Lineups cover all match folders.")
else:
    print(f"⚠️ Lineups: expected {n_match_folders}, got {n_matches_lineups}")


MATCH COVERAGE VALIDATION

Match report folders loaded: 58
Unique matches in player stats: 58
Unique matches in lineups: 55
✓ Player stats cover all match folders.
⚠️ Lineups: expected 58, got 55


In [40]:
# ============================================================================
# 1.10 TEAM VALIDATION
# ============================================================================

print("\n" + "=" * 100)
print("TEAM VALIDATION")
print("=" * 100)

if "team" in player_match_master_full.columns:
    print("\nTeam distribution:")
    print(player_match_master_full["team"].value_counts(dropna=False))
    
    # Check if non-Arsenal teams are included
    non_arsenal = player_match_master_full[
        ~player_match_master_full["team"].str.contains("Arsenal", case=False, na=False)
    ]
    
    if len(non_arsenal) > 0:
        print(f"\n⚠️ Found {len(non_arsenal)} rows with non-Arsenal teams")
        print("These will be filtered out next.")
    else:
        print("✓ Only Arsenal players in dataset.")
else:
    print("⚠️ No team column found.")


TEAM VALIDATION

Team distribution:
team
Arsenal    866
Name: count, dtype: int64
✓ Only Arsenal players in dataset.


In [41]:
# ============================================================================
# 1.11 CORE COLUMN COMPLETENESS
# ============================================================================

print("\n" + "=" * 100)
print("CORE COLUMN COMPLETENESS")
print("=" * 100)

core_cols = [
    "match_id", "date", "competition", "venue", "opponent",
    "team", "player", "pos", "min"
]

available_core_cols = [c for c in core_cols if c in player_match_master_full.columns]

core_missing = (
    player_match_master_full[available_core_cols]
    .isna()
    .sum()
    .reset_index()
)

core_missing.columns = ["column", "missing_count"]
core_missing["missing_pct"] = (
    core_missing["missing_count"] / len(player_match_master_full) * 100
).round(2)

core_missing = core_missing.sort_values("missing_count", ascending=False)

print(f"\nCore columns checked: {len(available_core_cols)}")
print(core_missing.to_string(index=False))

# Flag any columns with >5% missing data
high_missing = core_missing[core_missing["missing_pct"] > 5]
if len(high_missing) > 0:
    print(f"\n⚠️ Columns with >5% missing data:")
    for _, row in high_missing.iterrows():
        print(f"   {row['column']}: {row['missing_pct']:.1f}%")
else:
    print("\n✓ All core columns <5% missing data.")


CORE COLUMN COMPLETENESS

Core columns checked: 9
     column  missing_count  missing_pct
   match_id              0          0.0
       date              0          0.0
competition              0          0.0
      venue              0          0.0
   opponent              0          0.0
       team              0          0.0
     player              0          0.0
        pos              0          0.0
        min              0          0.0

✓ All core columns <5% missing data.


In [42]:
# ============================================================================
# 1.12 FILTER FOR ARSENAL PLAYERS ONLY
# ============================================================================

print("\n" + "=" * 100)
print("FILTERING FOR ARSENAL PLAYERS")
print("=" * 100)

rows_before = len(player_match_master_full)

if "team" in player_match_master_full.columns:
    player_match_master_full = player_match_master_full[
        player_match_master_full["team"].str.contains("Arsenal", case=False, na=False)
    ].copy()
    
    rows_after = len(player_match_master_full)
    rows_removed = rows_before - rows_after
    
    print(f"\nRows before filtering: {rows_before}")
    print(f"Rows after Arsenal filter: {rows_after}")
    print(f"Rows removed: {rows_removed}")
    
    if rows_removed > 0:
        print(f"✓ Filtered out non-Arsenal players")
    else:
        print("✓ All rows are Arsenal players")
else:
    print("⚠️ No team column to filter. Assuming all rows are Arsenal players.")


FILTERING FOR ARSENAL PLAYERS

Rows before filtering: 866
Rows after Arsenal filter: 866
Rows removed: 0
✓ All rows are Arsenal players


## 5. Merge Player-Match Data

In [43]:
# player_match_master_full is already the combined player + goalkeeper match data
# No additional merging needed for these tables

df_player_match = player_match_master_full.copy()

print(f"Player-match merged dataset shape: {df_player_match.shape}")
print(f"Columns: {df_player_match.columns.tolist()[:20]}...")

Player-match merged dataset shape: (866, 37)
Columns: ['match_id', 'match_report_url', 'date', 'competition', 'round', 'venue', 'opponent', 'result', 'match_report_name', 'team', 'player', 'shirtnumber', 'nation', 'pos', 'age', 'min', 'gls', 'ast', 'pk', 'pkatt']...


## 6. Deduplicate and Validate Dataset

In [44]:
print("\n" + "="*80)
print("DEDUPLICATION: REMOVING DUPLICATE PLAYER-MATCH COMBINATIONS")
print("="*80)

print(f"\nBefore deduplication:")
print(f"  Shape: {df_player_match.shape}")

# Count duplicates by match_id and player
duplicate_counts = (
    df_player_match
    .groupby(['match_id', 'player'])
    .size()
    .reset_index(name='count')
)
duplicates = duplicate_counts[duplicate_counts['count'] > 1]

print(f"  Duplicate player-match combinations found: {len(duplicates)}")
if len(duplicates) > 0:
    print(f"\n  Sample duplicates (first 10):")
    for idx, row in duplicates.head(10).iterrows():
        match_id = row['match_id']
        player = row['player']
        count = row['count']
        print(f"    {player} (Match {match_id}): {count} duplicate rows")

# Remove exact duplicates (same match_id, player, keep first occurrence)
df_player_match = (
    df_player_match
    .sort_values(['match_id', 'player'])
    .drop_duplicates(subset=['match_id', 'player'], keep='first')
    .reset_index(drop=True)
)

print(f"\nAfter deduplication:")
print(f"  Shape: {df_player_match.shape}")

# Verify deduplication succeeded
remaining_duplicates = (
    df_player_match
    .groupby(['match_id', 'player'])
    .size()
    .reset_index(name='count')
)
remaining_dups = remaining_duplicates[remaining_duplicates['count'] > 1]

if len(remaining_dups) == 0:
    print(f"  ✓ Deduplication successful - no remaining duplicates")
else:
    print(f"  ⚠️  WARNING: {len(remaining_dups)} duplicate combinations remain")

print(f"\n" + "="*80)
print("DATASET VALIDATION")
print("="*80)

# Sanity checks
n_matches = df_player_match['match_id'].nunique()
n_players = df_player_match['player'].nunique()
max_matches_per_player = df_player_match.groupby('player').size().max()
min_matches_per_player = df_player_match.groupby('player').size().min()
avg_matches_per_player = df_player_match.groupby('player').size().mean()

print(f"\nDataset dimensions:")
print(f"  Total rows (player-matches): {len(df_player_match)}")
print(f"  Unique matches: {n_matches}")
print(f"  Unique players: {n_players}")
print(f"  Average players per match: {len(df_player_match) / n_matches:.1f}")
print(f"  Average matches per player: {avg_matches_per_player:.1f}")
print(f"  Max matches for any player: {max_matches_per_player}")
print(f"  Min matches for any player: {min_matches_per_player}")
print(f"\n✓ Sanity check: {max_matches_per_player} ≤ 65 (expected for full season)")


DEDUPLICATION: REMOVING DUPLICATE PLAYER-MATCH COMBINATIONS

Before deduplication:
  Shape: (866, 37)
  Duplicate player-match combinations found: 0

After deduplication:
  Shape: (866, 37)
  ✓ Deduplication successful - no remaining duplicates

DATASET VALIDATION

Dataset dimensions:
  Total rows (player-matches): 866
  Unique matches: 58
  Unique players: 32
  Average players per match: 14.9
  Average matches per player: 27.1
  Max matches for any player: 56
  Min matches for any player: 1

✓ Sanity check: 56 ≤ 65 (expected for full season)


## 7. Calculate Rest Days

In [45]:
# Sort by player and date
df_rest = df_player_match.sort_values(['player', 'date']).copy()

# Calculate days since previous match for each player
df_rest['prev_date'] = df_rest.groupby('player')['date'].shift(1)
df_rest['rest_days'] = (df_rest['date'] - df_rest['prev_date']).dt.days

print("Rest days calculation:")
print(f"\n  Sample data with rest_days:")
sample_cols = ['player', 'date', 'prev_date', 'rest_days', 'match_id']
print(df_rest[sample_cols].head(15))

# Summary of rest days
print(f"\n  Rest days distribution:")
print(df_rest['rest_days'].describe())

# Count missing rest days (first appearance for each player)
missing_rest = df_rest['rest_days'].isna().sum()
print(f"\n  First appearances (missing rest_days): {missing_rest}")

Rest days calculation:

  Sample data with rest_days:
           player       date  prev_date  rest_days  match_id
788  Ayden Heaven 2024-10-30        NaT        NaN  da5bf839
650     Ben White 2024-08-17        NaT        NaN  c0e3342a
308     Ben White 2024-08-24 2024-08-17        7.0  4692171a
561     Ben White 2024-08-31 2024-08-24        7.0  a843d023
164     Ben White 2024-09-15 2024-08-31       15.0  17774a57
118     Ben White 2024-09-19 2024-09-15        4.0  108b9e41
728     Ben White 2024-09-22 2024-09-19        3.0  d7538020
30      Ben White 2024-10-19 2024-09-22       27.0  01e63a1f
293     Ben White 2024-10-22 2024-10-19        3.0  32f2220e
380     Ben White 2024-10-27 2024-10-22        5.0  68aa1099
696     Ben White 2024-11-02 2024-10-27        6.0  cc960c22
149     Ben White 2024-11-06 2024-11-02        4.0  16d01a57
218     Ben White 2024-11-10 2024-11-06        4.0  2874d56e
804     Ben White 2025-02-22 2024-11-10      104.0  e511e91c
712     Ben White 2025-02-26 20

## 8. Create Rest Categories

**Rest category definition** (standardized across all notebooks):
- First appearance: no previous appearance available
- Short rest: ≤4 days since previous appearance
- Moderate rest: 5–6 days
- Normal rest: ≥7 days

In [46]:
# Create rest categories
def categorize_rest(days):
    if pd.isna(days):
        return 'First appearance'
    elif days <= 4:
        return 'Short rest'
    elif days <= 6:
        return 'Moderate rest'
    else:
        return 'Normal rest'

df_rest['rest_category'] = df_rest['rest_days'].apply(categorize_rest)

print("Rest category distribution:")
rest_dist = df_rest['rest_category'].value_counts().sort_index()
print(rest_dist)
print(f"\nPercentage distribution:")
print(df_rest['rest_category'].value_counts(normalize=True).sort_index() * 100)

# Sample of rest categories
print(f"\nSample data with rest_category:")
sample_cols = ['player', 'date', 'rest_days', 'rest_category', 'match_id']
print(df_rest[sample_cols].head(20))

Rest category distribution:
rest_category
First appearance     32
Moderate rest       105
Normal rest         239
Short rest          490
Name: count, dtype: int64

Percentage distribution:
rest_category
First appearance     3.695150
Moderate rest       12.124711
Normal rest         27.598152
Short rest          56.581986
Name: proportion, dtype: float64

Sample data with rest_category:
           player       date  rest_days     rest_category  match_id
788  Ayden Heaven 2024-10-30        NaN  First appearance  da5bf839
650     Ben White 2024-08-17        NaN  First appearance  c0e3342a
308     Ben White 2024-08-24        7.0       Normal rest  4692171a
561     Ben White 2024-08-31        7.0       Normal rest  a843d023
164     Ben White 2024-09-15       15.0       Normal rest  17774a57
118     Ben White 2024-09-19        4.0        Short rest  108b9e41
728     Ben White 2024-09-22        3.0        Short rest  d7538020
30      Ben White 2024-10-19       27.0       Normal rest  01e63a1

## 9. Create Position Groups

Consolidate raw positions into 6 tactical groups for analysis.

In [47]:
def simplify_position(pos_str):
    """
    Map raw FBref-style position strings to consolidated tactical groups.

    Tactical groups:
    - Goalkeeper
    - Centre-back
    - Full-back
    - Central Midfielder
    - Attacking Midfielder/Winger
    - Forward
    - Other
    """
    if pd.isna(pos_str):
        return "Other"
    
    pos = str(pos_str).strip().upper()
    
    # Explicit multi-position mappings
    if pos in ["FW,MF", "MF,FW"]:
        return "Attacking Midfielder/Winger"
    
    if pos in ["DF,MF", "MF,DF"]:
        return "Central Midfielder"
    
    # Single-position mappings
    if pos == "GK":
        return "Goalkeeper"
    
    elif pos in ["CB", "DF"]:
        return "Centre-back"
    
    elif pos in ["LB", "RB"]:
        return "Full-back"
    
    elif pos in ["DM", "CM", "MF"]:
        return "Central Midfielder"
    
    elif pos in ["LM", "RM", "AM", "LW", "RW"]:
        return "Attacking Midfielder/Winger"
    
    elif pos == "FW":
        return "Forward"
    
    else:
        return "Other"


df_rest["position_group"] = df_rest["pos"].apply(simplify_position)

print("✓ Position groups created")

# Distribution summary
position_group_summary = (
    df_rest
    .groupby("position_group")
    .agg(
        n_player_matches=("match_id", "count"),
        n_players=("player", "nunique"),
        avg_minutes=("min", "mean"),
        total_minutes=("min", "sum")
    )
    .reset_index()
    .sort_values("n_player_matches", ascending=False)
)

print("\nPosition group distribution:")
print(position_group_summary.to_string(index=False))

print("\nRaw positions mapped to groups:")
for group in sorted(df_rest["position_group"].unique()):
    raw_positions = (
        df_rest.loc[df_rest["position_group"] == group, "pos"]
        .dropna()
        .unique()
    )
    print(f"  {group}: {sorted(raw_positions)}")

print("\nMulti-position labels identified:")
multi_pos = (
    df_rest[
        df_rest["pos"].astype(str).str.contains(",", na=False)
    ][["player", "pos", "position_group"]]
    .drop_duplicates()
    .sort_values(["pos", "player"])
)

if len(multi_pos) > 0:
    print(f"Found {len(multi_pos)} unique player-position combinations with multiple positions")
    print(multi_pos.to_string(index=False))
else:
    print("No multi-position players found")

print("\nPositions mapped to Other:")
other_positions = (
    df_rest[df_rest["position_group"] == "Other"]["pos"]
    .dropna()
    .value_counts()
)

if len(other_positions) > 0:
    print(other_positions.to_string())
else:
    print("✓ No raw positions mapped to Other")

✓ Position groups created

Position group distribution:
             position_group  n_player_matches  n_players  avg_minutes  total_minutes
         Central Midfielder               263         12    58.346008          15345
                    Forward               164         10    80.079268          13133
                Centre-back               163         11    68.361963          11143
                  Full-back               116          8    79.577586           9231
Attacking Midfielder/Winger               102         10    33.539216           3421
                 Goalkeeper                58          4    90.517241           5250

Raw positions mapped to groups:
  Attacking Midfielder/Winger: ['AM', 'FW,MF', 'LM', 'RM']
  Central Midfielder: ['CM', 'DF,MF', 'DM', 'MF']
  Centre-back: ['CB', 'DF']
  Forward: ['FW']
  Full-back: ['LB', 'RB']
  Goalkeeper: ['GK']

Multi-position labels identified:
Found 13 unique player-position combinations with multiple positions
          

## 10. Save Cleaned Datasets

In [ ]:
# Prepare final datasets for export
df_final = df_rest.copy()

# Select columns to keep
core_cols = [
    'match_id', 'date', 'player', 'pos', 'position_group',
    'min', 'rest_days', 'rest_category'
]

# Include all performance columns
performance_cols = [col for col in df_final.columns if col not in core_cols + ['prev_date']]

export_cols = core_cols + performance_cols
export_cols = [col for col in export_cols if col in df_final.columns]

df_export = df_final[export_cols].copy()

print("Export dataset prepared:")
print(f"  Shape: {df_export.shape}")
print(f"  Columns: {len(export_cols)} total")
print(f"  First 10 columns: {export_cols[:10]}")

# Save to processed directory
output_file = PROCESSED_PATH / 'arsenal_player_match_master.csv'
df_export.to_csv(output_file, index=False)
print(f"\n✓ Saved: {output_file}")

# Also save rest days separately for reference
rest_cols = ['match_id', 'date', 'player', 'rest_days', 'rest_category', 'position_group']
rest_cols = [col for col in rest_cols if col in df_final.columns]
df_rest_export = df_final[rest_cols].copy()

rest_file = PROCESSED_PATH / 'arsenal_player_rest_days.csv'
df_rest_export.to_csv(rest_file, index=False)
print(f"✓ Saved: {rest_file}")

# Save lineups master for formation visualization
if 'lineups_master' in dir():
    lineups_export = lineups_master.copy()
    lineups_file = PROCESSED_PATH / 'arsenal_lineups_master.csv'
    lineups_export.to_csv(lineups_file, index=False)
    print(f"✓ Saved: {lineups_file}")

print(f"\n" + "="*80)
print("DATA PREPARATION COMPLETE")
print("="*80)
print(f"\nNext steps:")
print(f"  1. Load processed dataset in Notebook 02")
print(f"  2. Use for performance exploration and fatigue analysis")
print(f"  3. Generate model-ready dataset for Notebook 03")

Export dataset prepared:
  Shape: (866, 40)
  Columns: 40 total
  First 10 columns: ['match_id', 'date', 'player', 'pos', 'position_group', 'min', 'rest_days', 'rest_category', 'match_report_url', 'competition']

✓ Saved: /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/processed/arsenal_player_match_master.csv
✓ Saved: /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/processed/arsenal_player_rest_days.csv

DATA PREPARATION COMPLETE

Next steps:
  1. Load processed dataset in Notebook 02
  2. Use for performance exploration and fatigue analysis
  3. Generate model-ready dataset for Notebook 03


## Data preparation summary

This notebook created the cleaned Arsenal 2024/25 player-match dataset used for the performance exploration and predictive modelling notebooks.

The final dataset contains 866 player-match observations across 58 matches and 32 Arsenal players. Duplicate goalkeeper rows were resolved so that each row represents one player in one match. Rest days were calculated at the player level by measuring the number of days since each player’s previous appearance. Rest categories were defined as:

- First appearance: no previous appearance available
- Short rest: ≤4 days
- Moderate rest: 5–6 days
- Normal rest: ≥7 days

Position labels were consolidated into six tactical groups:

- Goalkeeper
- Centre-back
- Full-back
- Central Midfielder
- Attacking Midfielder/Winger
- Forward

The processed datasets were saved to the `processed/` directory and will be used as inputs for Notebook 02.

## 11. VISUALIZATION SECTIONS MOVED TO NOTEBOOK 02

All visualization code has been refactored into reusable, parameterized functions and moved to **02_performance_exploration_fatigue_arsenal.ipynb**.

### Removed Sections:
1. **Section 11: Playing Time Analysis** → `plot_playing_time_by_position()` in Notebook 02
   - Creates 2x2 subplot grid (Goalkeepers, Defenders, Midfielders, Forwards)
   - Striped bars show estimated starting vs bench minutes
   - Reusable across any team via theme parameter

2. **Section 12: Formation Visualizations** → `plot_formation_visualization()` in Notebook 02
   - Uses mplsoccer to render formations on vertical pitch
   - Auto-detects formations from lineup data
   - Plots top N formations with player positions

3. **Section 13: Rest Days Distribution** → `plot_rest_distribution_discrete()` and `plot_player_rest_heatmap()` in Notebook 02
   - Two-panel congestion analysis (discrete bars + summary categories)
   - Calendar heatmap with 7 color-coded rest categories
   - Already executed in Notebook 02 Section 10 (cells 36-37)

### Why This Change?
- **Separation of Concerns**: Notebook 01 = Pure ETL, Notebook 02 = Exploration & Visualization
- **Reusability**: All functions accept team_name, team_slug, theme, output_path parameters
- **Maintainability**: Easier to update viz logic without affecting data processing pipeline
- **Scalability**: Same functions work for any team (Arsenal, Liverpool, City, etc.)

### Data Export Summary
This notebook successfully exports these processed files (sections 1-10):
- `arsenal_player_match_master.csv` - Full player-match data
- `arsenal_player_rest_days.csv` - Rest analysis subset
- `arsenal_player_match_model_ready.csv` - ML model preparation

**Status**: ✅ ETL pipeline complete. All analysis continues in Notebook 02.